<a href="https://colab.research.google.com/github/irungus/tree-species-ml-pipeline/blob/main/notebooks/01_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.0/478.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 107.3 MB/s eta 0:00:00


In [1]:
import pandas as pd # load and preprocess data
import os #manage file paths and directories.
import requests # sending HTTP requests to interact with web APIs or download content from the internet.
import io #Provides tools for working with I/O streams.
import matplotlib.pyplot as plt # Data visualization
import seaborn as sns# Data visualization
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import matplotlib.pyplot as plt
import ee
import ee
import time
from tqdm import tqdm

In [2]:
ee.Authenticate()
ee.Initialize(project='agfkenya')

In [3]:
url = "https://raw.githubusercontent.com/irungus/tree-species-ml-pipeline/main/data/raw/Data.csv"
df = pd.read_csv(url)

print(df.head())
print(f"Total points: {len(df)}")

# Working dataframe
feature_df = df.copy()

    uid     species  latitude  longitude
0  6042      Acacia -4.537429  39.139728
1  5988  Eucalyptus -4.522492  39.183680
2  5985  Eucalyptus -4.522487  39.183677
3  5982  Eucalyptus -4.522480  39.183657
4  5932  Eucalyptus -4.472281  39.189929
Total points: 3071


# Helper Functions

In [4]:
def extract_values(image, point):
    try:
        values = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point,
            scale=30,
            bestEffort=True
        )
        return values.getInfo()
    except:
        return {}

## FEATURE EXTRACTION (GEE + Python)

We’ll extract:

### 🌱 Vegetation Indices (Sentinel-2)
- NDVI  
- EVI  
- SAVI  

### 📡 SAR Features (Sentinel-1)
- VV  
- VH  
- VV/VH ratio  

### 🌡 Climate
- Temperature (mean)  
- Precipitation (mean)  

### ⛰ Terrain
- Elevation  
- Slope  

# Vegetation Indices

In [5]:
veg_results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Vegetation"):
    point = ee.Geometry.Point([row["longitude"], row["latitude"]])

    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate("2022-01-01", "2022-12-31")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .select(["B2", "B4", "B8"])
        .median()
    )

    ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")

    evi = s2.expression(
        "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
        {
            "NIR": s2.select("B8"),
            "RED": s2.select("B4"),
            "BLUE": s2.select("B2"),
        },
    ).rename("EVI")

    savi = s2.expression(
        "((NIR - RED) / (NIR + RED + 0.5)) * 1.5",
        {
            "NIR": s2.select("B8"),
            "RED": s2.select("B4"),
        },
    ).rename("SAVI")

    img = ee.Image.cat([ndvi, evi, savi])

    vals = extract_values(img, point)
    veg_results.append(vals)

    time.sleep(0.1)  # avoid API throttling

veg_df = pd.DataFrame(veg_results)
feature_df = pd.concat([feature_df, veg_df], axis=1)

feature_df.to_csv("vegetation_done.csv", index=False)
print("✅ Vegetation complete")

Vegetation: 100%|██████████| 3071/3071 [47:48<00:00,  1.07it/s]

✅ Vegetation complete


# SAR Features

In [6]:
sar_results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="SAR"):
    point = ee.Geometry.Point([row["longitude"], row["latitude"]])

    s1 = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(point)
        .filterDate("2022-01-01", "2022-12-31")
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select(["VV", "VH"])
        .median()
    )

    ratio = s1.select("VV").divide(s1.select("VH")).rename("VV_VH_ratio")

    img = ee.Image.cat([
        s1.select("VV"),
        s1.select("VH"),
        ratio
    ])

    vals = extract_values(img, point)
    sar_results.append(vals)

    time.sleep(0.1)

sar_df = pd.DataFrame(sar_results)
feature_df = pd.concat([feature_df, sar_df], axis=1)

feature_df.to_csv("sar_done.csv", index=False)
print("✅ SAR complete")

SAR: 100%|██████████| 3071/3071 [21:16<00:00,  2.41it/s]

✅ SAR complete


# Terrain Features

In [7]:
terrain_results = []

dem = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(dem)

terrain_img = ee.Image.cat([
    dem.rename("elevation"),
    slope.rename("slope")
])

for i, row in tqdm(df.iterrows(), total=len(df), desc="Terrain"):
    point = ee.Geometry.Point([row["longitude"], row["latitude"]])

    vals = extract_values(terrain_img, point)
    terrain_results.append(vals)

terrain_df = pd.DataFrame(terrain_results)
feature_df = pd.concat([feature_df, terrain_df], axis=1)

feature_df.to_csv("terrain_done.csv", index=False)
print("✅ Terrain complete")

Terrain: 100%|██████████| 3071/3071 [05:17<00:00,  9.66it/s]

✅ Terrain complete


# Climate Features

In [8]:
climate_results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Climate"):
    point = ee.Geometry.Point([row["longitude"], row["latitude"]])

    climate = (
        ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY")
        .filterBounds(point)
        .filterDate("2022-01-01", "2022-12-31")
        .mean()
    )

    vals = extract_values(climate, point)
    climate_results.append(vals)

    time.sleep(0.1)

climate_df = pd.DataFrame(climate_results)
feature_df = pd.concat([feature_df, climate_df], axis=1)

feature_df.to_csv("final_features.csv", index=False)
print("✅ Climate complete")

Climate:   0%|          | 0/3071 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for ECMWF/ERA5_LAND/MONTHLY! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/ECMWF_ERA5_LAND_MONTHLY

  warnings.warn(warning, category=DeprecationWarning)
Climate: 100%|██████████| 3071/3071 [26:34<00:00,  1.93it/s]


✅ Climate complete


In [9]:
if os.path.exists("data/processed/vegetation.csv"):
    feature_df = pd.read_csv("data/processed/vegetation.csv")